# XGBoost PU Bagging — KDE 核密度加权抽样（重要性采样 II）

**实验动机：** 与实验一相同——[Fe/H] > -1.2 的候选星中有相当一部分 N 丰度较低的"误判"，怀疑标准 PU 的等概率负采样让模型学会了用物理参数区分正负的捷径。

**方法：**
1. 对正样本与 U 池分别拟合 **3D 高斯核密度** f_pos / f_unl（(teff, logg, feh) 空间）
2. 以重要性权重 **w = f_pos / f_unl** 对 U 加权采样：抬高"正样本密度相对更高"区域的采样概率，压低"只有 U、没有正样本"区域的概率
3. 等价于用**密度比做协变量漂移校正**，让负样本的物理参数分布逼近正样本

**与实验一的区别：** 实验一是"硬匹配"（每个正样本找物理最近邻）；本实验是"软加权"（密度比连续重加权，不显式配对），两者互为印证。

In [ ]:
# 共享数据加载 + 导入重要性采样模块

import sys, os, time
from pathlib import Path

# 定位项目根目录（向上查找直到同时存在 ML/ 与 Data/）
_PROJECT_ROOT = Path(os.getcwd())
for _ in range(5):
    if (_PROJECT_ROOT / "ML").exists() and (_PROJECT_ROOT / "Data").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent
for p in (str(_PROJECT_ROOT), str(_PROJECT_ROOT / "XGB")):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 10})
import warnings
warnings.filterwarnings("ignore")

# 核心逻辑（采样器 + PU Bagging + 评估）统一在 XGB/importance_sampling.py
import importance_sampling as imp

t0 = time.time()
S = imp.load_and_split()
X_clean = S["X_clean"]
stars_clean = S["stars_clean"]
common_wave = S["common_wave"]
X_all = S["X_spec_scaled"]
X_tr = S["X_tr"]
X_te = S["X_te"]
y_tr = S["y_tr"]
y_te = S["y_te"]
n_pos_tr = S["n_pos_tr"]
unl_tr_idx = S["unl_tr_idx"]
tr_idx = S["tr_idx"]

feh_all = stars_clean["feh"].values.astype(float)

print(f"数据加载完成 ({time.time()-t0:.0f}s)")
print(f"总样本 {len(S['y_all']):,} | 训练 {len(tr_idx):,} (P={n_pos_tr}, U={len(unl_tr_idx):,}) | 测试 {len(S['test_idx']):,}")
print(f"物理参数列: teff/logg/feh（全样本无缺失）")


## 1. 构造 KDE 加权采样器

密度比 w = f_pos / f_unl 是协变量漂移校正的标准重要性权重。ESS（有效样本量）越小，说明权重越集中在少数 U 上，即物理分布被拉得越接近正样本。

In [ ]:
# 标准化物理参数 + 构造"KDE 核密度加权"采样器

name = "kde"
label = "KDE 核密度加权"

phys_all, mu, sd = imp.standardize_physics(stars_clean)
phys_tr = phys_all[tr_idx]
phys_pos = phys_tr[np.where(y_tr == 1)[0]]   # 训练正样本 (n_pos_tr, 3)
phys_unl = phys_tr[unl_tr_idx]               # 训练 U (n_unl, 3)

w, ess = imp.build_kde_weights(phys_pos, phys_unl, unl_tr_idx)
method_sampler = imp.WeightedSampler(unl_tr_idx, w, imp.random_seed)

print(f"物理参数标准化: mu(teff/logg/feh) = {mu.round(1)}, sd = {sd.round(2)}")
print(f"KDE 权重: max/median = {np.max(w)/np.median(w):.1f}  有效样本量 ESS = {ess:.0f} / {len(unl_tr_idx):,}")
print(f"权重集中度说明: ESS 越小，加权采样越'集中'在少数 U 上（物理分布越接近正样本）")


## 2. 采样分布诊断（[Fe/H]）

关键检验：KDE 加权是否把负样本的有效 [Fe/H] 分布拉向正样本分布。橙色曲线是权重重加权的直方图，应明显向红色（正样本）靠拢。

In [ ]:
# 诊断：KDE 加权是否把负样本的 [Fe/H] 分布"拉向"正样本

feh_pos = feh_all[tr_idx[np.where(y_tr == 1)[0]]]
feh_unl = feh_all[tr_idx[unl_tr_idx]]

bins = np.linspace(-2.6, -0.6, 41)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hist(feh_unl, bins=bins, density=True, alpha=0.45, color="steelblue",
        label=f"U 池（等概率采样, n={len(feh_unl):,}）")
ax.hist(feh_unl, bins=bins, weights=w, density=True, alpha=0.65, color="darkorange",
        label="KDE 加权后（密度比重加权）")
ax.hist(feh_pos, bins=bins, density=True, alpha=0.8, color="crimson",
        label=f"已知 CN 正样本（n={len(feh_pos)}）")
ax.axvline(-1.2, color="k", linestyle="--", linewidth=1.0, label="[Fe/H] = -1.2")
ax.set_xlabel("[Fe/H]")
ax.set_ylabel("密度")
ax.set_title("物理参数对齐：KDE 加权后负样本 [Fe/H] 分布 vs 正样本")
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"U 池(等概率)       中位 feh={np.median(feh_unl):.3f}  feh>-1.2 占比={np.mean(feh_unl > -1.2)*100:.1f}%")
print(f"KDE 加权(有效分布) 中位 feh={np.average(feh_unl, weights=w):.3f}  feh>-1.2 占比={np.sum(w[feh_unl > -1.2])*100:.1f}%")
print(f"正样本             中位 feh={np.median(feh_pos):.3f}  feh>-1.2 占比={np.mean(feh_pos > -1.2)*100:.1f}%")


## 3. 运行 PU Bagging（T=500）

在 KDE 加权采样器与标准等概率采样器上分别跑 T=500，其余完全一致。

In [ ]:
# 运行 PU Bagging（重要性采样 vs 标准等概率采样）

T = 500

print(f"[{name}] {label} (T={T}) ...")
te_method, p_method = imp.run_pu_bagging(
    X_tr, X_te, X_all, y_tr, y_te, n_pos_tr, method_sampler, T,
    label=name, report_every=100)

print()
print("[base] 标准 PU 等概率采样 (T=500) ...")
te_base, p_base = imp.run_pu_bagging(
    X_tr, X_te, X_all, y_tr, y_te, n_pos_tr, imp.UniformSampler(unl_tr_idx), T,
    label="base", report_every=100)

res_method = imp.metrics(y_te, te_method)
res_base = imp.metrics(y_te, te_base)

print()
print(f"{label:28s}  ROC={res_method['roc']:.4f}  PR={res_method['pr']:.4f}  "
      f"P@50={res_method['p50']:.4f}  P@100={res_method['p100']:.4f}")
print(f"{'标准 PU(等概率)':28s}  ROC={res_base['roc']:.4f}  PR={res_base['pr']:.4f}  "
      f"P@50={res_base['p50']:.4f}  P@100={res_base['p100']:.4f}")


## 4. 结果对比（指标 + PR/ROC 曲线）

In [ ]:
# 结果对比：指标表 + PR/ROC 曲线叠加

from sklearn.metrics import precision_recall_curve, roc_curve, auc

comp = pd.DataFrame({
    "采样方式": ["标准 PU (等概率)", label],
    "ROC-AUC": [res_base["roc"], res_method["roc"]],
    "PR-AUC": [res_base["pr"], res_method["pr"]],
    "P@50": [res_base["p50"], res_method["p50"]],
    "P@100": [res_base["p100"], res_method["p100"]],
})
print(comp.to_string(index=False))

base_rate = y_te.sum() / len(y_te)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

pr_base, rc_base, _ = precision_recall_curve(y_te, te_base)
pr_m, rc_m, _ = precision_recall_curve(y_te, te_method)
ax1 = axes[0]
ax1.plot(rc_base, pr_base, "b-", linewidth=2, label=f"标准 PU (AP={res_base['pr']:.3f})")
ax1.plot(rc_m, pr_m, "darkorange", linewidth=2, label=f"{label} (AP={res_method['pr']:.3f})")
ax1.axhline(base_rate, color="gray", linestyle="--", linewidth=1.0, label=f"Random ({base_rate:.3f})")
ax1.set_xlabel("Recall"); ax1.set_ylabel("Precision")
ax1.set_title("Precision-Recall Curve")
ax1.legend(fontsize=9); ax1.grid(alpha=0.2)
ax1.set_xlim(0, 1.02); ax1.set_ylim(0, 1.02)

fpr_base, tpr_base, _ = roc_curve(y_te, te_base)
fpr_m, tpr_m, _ = roc_curve(y_te, te_method)
ax2 = axes[1]
ax2.plot(fpr_base, tpr_base, "darkred", linewidth=2, label=f"标准 PU (AUC={auc(fpr_base, tpr_base):.3f})")
ax2.plot(fpr_m, tpr_m, "darkorange", linewidth=2, label=f"{label} (AUC={auc(fpr_m, tpr_m):.3f})")
ax2.plot([0, 1], [0, 1], "gray", linestyle="--", linewidth=1.0)
ax2.set_xlabel("False Positive Rate"); ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve")
ax2.legend(fontsize=9); ax2.grid(alpha=0.2)
ax2.set_xlim(0, 1.02); ax2.set_ylim(0, 1.02)

fig.suptitle("标准 PU vs " + label, fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 5. 已知 CN 星标定阈值 + 候选体 [Fe/H] 分解

直接观察重要性采样是否改变候选体在 [Fe/H] > -1.2 与 <= -1.2 两个区域的分布。

In [ ]:
# 已知 CN 星标定阈值 + 候选体 [Fe/H] 分解

recall_quantile = 0.05
thr_method, ncand_method, nknown_method = imp.threshold_candidates(stars_clean, p_method, recall_quantile)
thr_base, ncand_base, nknown_base = imp.threshold_candidates(stars_clean, p_base, recall_quantile)

def feh_split(prob, thr):
    cand = (stars_clean["label"].values == -1) & (prob >= thr)
    f = feh_all[cand]
    return dict(n=int(cand.sum()), gt=int((f > -1.2).sum()), le=int((f <= -1.2).sum()),
                median=float(np.median(f)))

fs_method = feh_split(p_method, thr_method)
fs_base = feh_split(p_base, thr_base)

print(f"标定阈值 (保留~{(1-recall_quantile)*100:.0f}% 已知 CN 星):")
print(f"  标准 PU(等概率): thr={thr_base:.4f}  候选={ncand_base}  已知CN>=thr={nknown_base}")
print(f"  {label}: thr={thr_method:.4f}  候选={ncand_method}  已知CN>=thr={nknown_method}")
print()
print("候选体 [Fe/H] 分解:")
for nm, fs in [("标准 PU(等概率)", fs_base), (label, fs_method)]:
    print(f"  {nm:22s} n={fs['n']:5d}  feh>-1.2: {fs['gt']:5d}  feh<=-1.2: {fs['le']:5d}  中位feh={fs['median']:.3f}")

outpath = str(_PROJECT_ROOT / "XGB" / "XGB_PU_kde_candidates_threshold.csv")
n_exp = imp.export_candidates(stars_clean, p_method, thr_method, outpath)
print(f"\n候选体已导出: {outpath}  ->  {n_exp} 颗")


## 6. 结论

**KDE 核密度加权抽样 vs 标准等概率采样：**

1. **物理参数对齐是否生效**：看第 2 节直方图，加权后的 [Fe/H] 分布应更接近正样本
2. **是否影响识别**：对比第 4 节 ROC-AUC / PR-AUC / P@50 / P@100
3. **是否缓解 feh>-1.2 误判**：看第 5 节候选体 [Fe/H] 分解
4. **与实验一对比**：若两种重要性采样结论一致（同升或同降），则结论更稳健；若不一致，说明匹配的"紧/松"程度对结果敏感
5. **调参**：`gaussian_kde` 的带宽可调（如 `bw_method` 缩放因子）；带宽越小，权重越尖锐、对齐越激进